In [4]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# --- CONFIGURACIÓN DE RUTA ---
PATH = r"C:\Users\pauli\Downloads\Subscriber_Collection_Report"
FILE_INVOICES = os.path.join(PATH, "invoiceDetailList_export (6).csv")
FILE_ALLOCATIONS = os.path.join(PATH, "allocationList_export (61).csv")
OUTPUT_FILE = os.path.join(PATH, "Subscriber_Collection_Report.xlsx")
FECHA_CORTE = pd.to_datetime("2026-03-01")

def ejecutar_maestro_total_con_resumen():
    try:
        # 1. CARGA Y LIMPIEZA
        df_inv = pd.read_csv(FILE_INVOICES)
        df_allo = pd.read_csv(FILE_ALLOCATIONS)

        clean_acc = lambda x: str(x).strip().split('.')[0] if pd.notna(x) else "NAN"
        df_inv['Account'] = df_inv['Account'].apply(clean_acc)
        df_allo['Account'] = df_allo.iloc[:, 3].apply(clean_acc)
        
        inv_col = df_inv.columns[1]
        col_usage, col_addr, col_city, col_zip = df_allo.columns[13], df_allo.columns[18], df_allo.columns[20], df_allo.columns[22]

        for col in ['Due', 'Invoice Total', 'Paid']:
            df_inv[col] = pd.to_numeric(df_inv[col], errors='coerce').fillna(0)
        df_allo[col_usage] = pd.to_numeric(df_allo[col_usage], errors='coerce').fillna(0)

        # 2. SEGMENTACIÓN
        status_col, reason_col = df_allo.columns[8], df_allo.columns[17]
        df_allo['Case Segment'] = df_allo.apply(lambda r: "Case 3: Delinquent Portfolio" if "Delinquent" in str(r[reason_col]) else ("Case 1: Loyal Subscribers" if str(r[status_col]) in ['Active', 'Pending Activation', 'Reserved in Facility'] else "Case 2: Cancelled Subscribers"), axis=1)
        
        df_inv['Date'] = pd.to_datetime(df_inv['Date'], errors='coerce')
        df_inv['Interest_10'] = (df_inv['Due'] * 0.10 / 365) * (FECHA_CORTE - df_inv['Date'].apply(lambda x: x.replace(day=16) if pd.notna(x) else x)).dt.days.clip(lower=0)

        # 3. PROCESAMIENTO REPORTE MENSUAL (CON DESGLOSE)
        rango_meses = pd.date_range(start="2025-09-01", end="2026-02-01", freq='MS')
        reporte_rows = []
        for acc in df_inv['Account'].unique():
            info = df_allo[df_allo['Account'] == acc]
            if info.empty: continue
            d_acc = df_inv[df_inv['Account'] == acc]
            fila = {'Account': acc, 'Segment': info['Case Segment'].values[0], 'Status': info[status_col].values[0], 'Billing Address': info[col_addr].values[0]}
            m_count = 0
            for m in rango_meses:
                lbl = m.strftime('%b %Y')
                m_data = d_acc[(d_acc['Date'].dt.month == m.month) & (d_acc['Date'].dt.year == m.year)]
                if not m_data.empty:
                    d_val = m_data['Due'].sum()
                    fila[f'Inv# ({lbl})'] = ", ".join(m_data[inv_col].astype(str).unique())
                    fila[f'Due ({lbl})'] = d_val
                    if d_val > 0.01: m_count += 1
                else:
                    fila[f'Inv# ({lbl})'], fila[f'Due ({lbl})'] = "", 0
            fila['Months Delinquent'] = m_count
            reporte_rows.append(fila)
        df_rep_final = pd.DataFrame(reporte_rows)

        # 4. FILTROS PARA CONSOLIDACIÓN Y RESUMEN ESTRATÉGICO
        filtro_critico = (
            ((df_rep_final['Segment'] == "Case 2: Cancelled Subscribers") & (df_rep_final['Months Delinquent'] > 0)) | 
            ((df_rep_final['Segment'] == "Case 3: Delinquent Portfolio") & (df_rep_final['Months Delinquent'].isin([3, 4, 5, 6])))
        )
        acc_prioridad = df_rep_final[filtro_critico]['Account'].unique()

        # KPIs para la tablita nueva
        sum_principal_due = df_inv[df_inv['Account'].isin(acc_prioridad)]['Due'].sum()
        sum_usage_critico = df_allo[df_allo['Account'].isin(acc_prioridad)][col_usage].sum()

        # Generar Hoja Horizontal
        df_priority = df_inv[df_inv['Account'].isin(acc_prioridad) & (df_inv['Due'] > 0.01)].merge(df_allo[['Account', col_usage, col_addr, col_city, col_zip]], on='Account', how='left')
        consol_rows = []
        for acc in acc_prioridad:
            details = df_priority[df_priority['Account'] == acc]
            if details.empty or details['Due'].sum() <= 0.01: continue
            row_data = {'Account Number': acc, 'Estimated Annual Usage': details[col_usage].iloc[0], 'Billing Address': details[col_addr].iloc[0], 'Total Account Due': details['Due'].sum()}
            for i, (_, inv_row) in enumerate(details.sort_values('Date').iterrows(), 1):
                row_data[f'Invoice_{i}'], row_data[f'Due_{i}'] = inv_row[inv_col], inv_row['Due']
            consol_rows.append(row_data)
        df_horizontal = pd.DataFrame(consol_rows)

        # 5. EXPORTACIÓN COMPLETA
        with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
            ws_dash = writer.book.create_sheet("Executive Dashboard", 0)
            navy_fill, gold_fill = PatternFill(start_color="1B2631", end_color="1B2631", fill_type="solid"), PatternFill(start_color="D4AF37", end_color="D4AF37", fill_type="solid")
            white_font = Font(color="FFFFFF", bold=True)
            thin_border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))

            # --- TABLITA DE RESUMEN ESTRATÉGICO (DERECHA) ---
            ws_dash.cell(row=2, column=8, value="STRATEGIC RECOVERY SUMMARY").font = Font(bold=True, size=11)
            ws_dash.cell(row=3, column=8, value="KPI Description").fill = navy_fill
            ws_dash.cell(row=3, column=8, value="KPI Description").font = white_font
            ws_dash.cell(row=3, column=9, value="Value").fill = navy_fill
            ws_dash.cell(row=3, column=9, value="Value").font = white_font
            
            ws_dash.cell(row=4, column=8, value="Total Critical Debt (Case 2 & 3)").border = thin_border
            d_cell = ws_dash.cell(row=4, column=9, value=sum_principal_due)
            d_cell.number_format, d_cell.font, d_cell.border = '"$"#,##0.00', Font(bold=True, color="922B21"), thin_border
            
            ws_dash.cell(row=5, column=8, value="Total Critical Usage (kWh)").border = thin_border
            u_cell = ws_dash.cell(row=5, column=9, value=sum_usage_critico)
            u_cell.number_format, u_cell.font, u_cell.border = '#,##0', Font(bold=True), thin_border

            # --- DASHBOARD PRINCIPAL ---
            curr_row = 2
            ws_dash.cell(row=curr_row, column=2, value="EXECUTIVE COLLECTION DASHBOARD").font = Font(bold=True, size=14)
            curr_row += 2

            def write_dash_table(title, df_filter):
                nonlocal curr_row
                if df_filter.empty: return
                ws_dash.cell(row=curr_row, column=2, value=title).font = Font(bold=True, size=12)
                curr_row += 1
                headers = ["Month", "Principal Due", "Interest (10%)", "Total Recoverable", "Usage (kWh)"]
                for i, h in enumerate(headers):
                    c = ws_dash.cell(row=curr_row, column=i+2, value=h)
                    c.fill = navy_fill if i == 0 else gold_fill
                    c.font = white_font
                curr_row += 1
                df_sub = df_inv[df_inv['Account'].isin(df_filter['Account'])]
                for m in rango_meses:
                    m_d = df_sub[(df_sub['Date'].dt.month == m.month) & (df_sub['Date'].dt.year == m.year)]
                    debtors = m_d[m_d['Due'] > 0.01]
                    if not debtors.empty:
                        u_sum = df_allo[df_allo['Account'].isin(debtors['Account'].unique())][col_usage].sum()
                        ws_dash.cell(row=curr_row, column=2, value=m.strftime('%B %Y'))
                        ws_dash.cell(row=curr_row, column=3, value=debtors['Due'].sum()).number_format = '"$"#,##0.00'
                        ws_dash.cell(row=curr_row, column=4, value=debtors['Interest_10'].sum()).number_format = '"$"#,##0.00'
                        ws_dash.cell(row=curr_row, column=5, value=debtors['Due'].sum() + debtors['Interest_10'].sum()).number_format = '"$"#,##0.00'
                        ws_dash.cell(row=curr_row, column=6, value=u_sum).number_format = '#,##0'
                        curr_row += 1
                curr_row += 2

            write_dash_table("CASE 1: LOYAL SUBSCRIBERS", df_rep_final[df_rep_final['Segment'] == "Case 1: Loyal Subscribers"])
            write_dash_table("CASE 2: CANCELLED SUBSCRIBERS", df_rep_final[df_rep_final['Segment'] == "Case 2: Cancelled Subscribers"])
            for n in range(1, 7):
                write_dash_table(f"CASE 3 - {n} MONTH(S)", df_rep_final[(df_rep_final['Segment'] == "Case 3: Delinquent Portfolio") & (df_rep_final['Months Delinquent'] == n)])

            # --- TODAS LAS HOJAS RESTAURADAS ---
            df_horizontal.to_excel(writer, sheet_name='Account Consolidation', index=False)
            df_priority.to_excel(writer, sheet_name='Collection Priority Detail', index=False)
            df_rep_final.to_excel(writer, sheet_name='Reporte Mensual Info', index=False)

        print(f"✅ EXCEL COMPLETO Y RESTAURADO EN: {OUTPUT_FILE}")

    except Exception as e: print(f"❌ ERROR: {e}")

ejecutar_maestro_total_con_resumen()

✅ EXCEL COMPLETO Y RESTAURADO EN: C:\Users\pauli\Downloads\Subscriber_Collection_Report\Subscriber_Collection_Report.xlsx
